# Legacy RFGAP Comparison

This notebook compares the legacy `reference_rfgap.RFGAP` implementation with the current `LeafEncoder` API on Iris.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find the project root containing pyproject.toml and src/.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from forestkernel import LeafEncoder
from forestkernel.reference_rfgap import RFGAP


In [ ]:
seed = 42
weight_scheme = "gap"
legacy_prox_method = "rfgap"

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.2,
    stratify=iris.target,
    random_state=seed,
)


In [ ]:
legacy = RFGAP(
    prediction_type="classification",
    prox_method=legacy_prox_method,
    matrix_type="sparse",
    non_zero_diagonal=False,
    symm_mode=None,
    max_normalize=False,
    model_type="rf",
    n_estimators=100,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=seed,
)
legacy.fit(X_train, y_train)
K_legacy = legacy.get_proximities()
K_legacy_test = legacy.prox_extend(X_test)


In [ ]:
forest = legacy
# Use an independently fitted forest with matching settings for the current API.
# This checks implementation-level behavior, not bitwise equality of shared estimator state.
from sklearn.ensemble import RandomForestClassifier

current = LeafEncoder(
    forest=RandomForestClassifier(
        n_estimators=100,
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=seed,
    ),
    weight_scheme=weight_scheme,
).fit(X_train, y_train)

K_current = current.kernel()
K_current_test = current.kernel_extend(X_test)


In [ ]:
legacy_dense = K_legacy.toarray()
current_dense = K_current.toarray()

print(f"legacy train kernel : {K_legacy.shape}, nnz={K_legacy.nnz}")
print(f"current train kernel: {K_current.shape}, nnz={K_current.nnz}")
print(f"legacy test block   : {K_legacy_test.shape}")
print(f"current test block  : {K_current_test.shape}")
print(f"max |legacy - current|: {np.max(np.abs(legacy_dense - current_dense)):.3e}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

for ax, matrix, title in [
    (axes[0], legacy_dense, "legacy RFGAP"),
    (axes[1], current_dense, "LeafEncoder GAP"),
    (axes[2], np.abs(legacy_dense - current_dense), "absolute difference"),
]:
    im = ax.imshow(matrix, cmap="viridis", aspect="auto")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)

plt.show()
